In [0]:
print("Hello World!")

## Import Libraries + Data

In [0]:
import numpy as np
import pandas as pd
pd.set_option('display.max_columns', None)
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm # colormaps
# plt.style.use("dark_background")
import seaborn as sns
import math

from sklearn.compose import ColumnTransformer
from sklearn.cluster import AgglomerativeClustering, KMeans
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression, Ridge, LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import r2_score, mean_absolute_error,mean_squared_error, silhouette_score, silhouette_samples, confusion_matrix, classification_report
from sklearn.model_selection import train_test_split, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler, PolynomialFeatures, LabelEncoder, MinMaxScaler
from sklearn.svm import SVR
# import xgboost as xgb

import plotly.express as px
import warnings
warnings.filterwarnings('ignore')

In [0]:
# Install dependencies as needed:
# !pip install kagglehub[pandas-datasets]
# !pip install tensorflow

In [0]:
import kagglehub
from kagglehub import KaggleDatasetAdapter

# Set the path to the file you'd like to load
file_path = "WA_Fn-UseC_-Telco-Customer-Churn.csv"

# Load the latest version
df_data = kagglehub.load_dataset(
  KaggleDatasetAdapter.PANDAS,
  "blastchar/telco-customer-churn",
  file_path,
  # Provide any additional arguments like 
  # sql_query or pandas_kwargs. See the 
  # documenation for more information:
  # https://github.com/Kaggle/kagglehub/blob/main/README.md#kaggledatasetadapterpandas
)

df_data.head()

In [0]:
%skip
df_data = pd.read_csv("/kaggle/input/datasets/blastchar/telco-customer-churn/WA_Fn-UseC_-Telco-Customer-Churn.csv")
df_data.head()

## EDA, Data Visualization

In [0]:
df = df_data.copy()
df.drop('customerID', axis = 'columns', inplace = True)
df.dtypes

In [0]:
df1 = df[df.TotalCharges!= ' ']
df1.shape

In [0]:
df1.TotalCharges = pd.to_numeric(df1.TotalCharges)
df1.dtypes

In [0]:
# Identify categorical and numerical columns
categorical_cols = df1.select_dtypes(include=['object', 'category']).columns
numerical_cols = df1.select_dtypes(include=['int64', 'float64']).columns
categorical_cols, numerical_cols

In [0]:
# colors
available_colors = list(mpl.colors.cnames.keys())

available_colors = [
    'orangered', 'blue', 'green', 'orange', 'crimson', 'purple',
    'yellow', 'yellowgreen', 'pink', 'cyan', 'magenta', 'lime', 'teal',
    'navy', 'skyblue', 'indigo', 'blueviolet', 'beige',
    'plum', 'orchid', 'crimson', 'darkgreen'
]
# number of plots
n = len(categorical_cols)

# grid size (auto)
cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = axes.flatten()

for i, col in enumerate(categorical_cols):
    sns.countplot(
        data=df, 
        x=col, 
        ax=axes[i], 
        color=available_colors[i % len(available_colors)]
    )
    axes[i].set_title(col)

# remove extra empty plots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [0]:
import math
import matplotlib.pyplot as plt
import seaborn as sns

# colors
available_colors = [
    '#FF6B6B', '#4ECDC4', '#45B7D1', '#FFA07A',
    '#98D8C8', '#F7DC6F', '#BB8FCE', '#85C1E9',
    '#F1948A', '#52BE80', '#5DADE2', '#AF7AC5'
]

# number of plots
n = len(numerical_cols)

# grid size
cols = 3
rows = math.ceil(n / cols)

fig, axes = plt.subplots(rows, cols, figsize=(5*cols, 4*rows))
axes = axes.flatten()

for i, col in enumerate(numerical_cols):
    sns.histplot(
        data=df,
        x=col,
        ax=axes[i],
        color=available_colors[i % len(available_colors)],
        kde=True   # optional
    )
    axes[i].set_title(col)

# remove extra empty plots
for j in range(i+1, len(axes)):
    fig.delaxes(axes[j])

plt.tight_layout()
plt.show()

In [0]:
tenure_churn_no = df1[df1.Churn == 'No'].tenure
tenure_churn_yes = df1[df1.Churn == 'Yes'].tenure

plt.hist([tenure_churn_yes, tenure_churn_no], label=['Churn=Yes','Churn=No'])
plt.legend()

In [0]:
tc_churn_no = df1[df1.Churn == 'No'].TotalCharges
tc_churn_yes = df1[df1.Churn == 'Yes'].TotalCharges

plt.hist([tc_churn_yes, tc_churn_no], label=['Churn=Yes','Churn=No'])
plt.legend()

## Null values, Duplicates, Encoding

In [0]:
# Create the LabelEncoder instance
le = LabelEncoder()

# Dictionary to store encoders for each column
label_encoders = {}

# Loop through each categorical column and encode
for col in categorical_cols:
    le = LabelEncoder()
    df1[col + '_encoded'] = le.fit_transform(df1[col])
    label_encoders[col] = le  # Save encoder for later inverse transform


In [0]:
# ✅ Get mapping for churn column only
churn_mapping = dict(zip(label_encoders['Churn'].classes_,
                         label_encoders['Churn'].transform(label_encoders['Churn'].classes_)))

print("Churn Mapping (from → to):", churn_mapping)

In [0]:
df2 = df1.drop(categorical_cols, axis = 1)
df2.head()

In [0]:
df2.dtypes

In [0]:
# Create the scaler (default range 0 to 1)
scaler = MinMaxScaler()

# Fit and transform the numerical columns
df_scaled = pd.DataFrame(
    scaler.fit_transform(df2),
    columns=df2.columns
)

df_scaled.head()

## Model fitting, Evaluation

In [0]:
X = df_scaled.drop('Churn_encoded', axis = 1)
y = df_scaled['Churn_encoded']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size = 0.2, random_state = 42)

In [0]:
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [0]:
len(X_train.columns)

In [0]:
import tensorflow as tf
from tensorflow import keras

model = keras.Sequential([
    keras.layers.Dense(16, input_shape = (19,), activation = 'relu'),
    keras.layers.Dense(8, activation = 'relu'),
    keras.layers.Dense(1, activation = 'sigmoid')
    
])

model.compile(optimizer = 'adam',
              loss = 'binary_crossentropy',
              metrics = ['accuracy']
             )

model.fit(X_train, y_train, epochs = 20)

In [0]:
model.evaluate(X_test, y_test)

In [0]:
y_pred = model.predict(X_test)
y_pred[:10]

In [0]:
y_test[:10]

In [0]:
y_output = [1 if x >= 0.5 else 0 for x in y_pred]
y_output[:10]

In [0]:
print(classification_report(y_test, y_output))

In [0]:
cm = tf.math.confusion_matrix(labels = y_test, predictions = y_output)

plt.figure(figsize = (6, 4))
sns.heatmap(cm, annot = True, fmt = 'd')
plt.xlabel('Predicted')
plt.ylabel('Actual')

In [0]:
print('precision 0:', 944 / (944 + 209), 'precision 1:', 165 / (165 + 89))
print('recall 0:', 944 / (944 + 89), 'recall 1:', 165 / (165 + 209))